In [2]:
"""
BioLogic .mpr Batch Analysis Tool
===================================
Only uses raw columns: t, E, I
No reliance on cycle number, half cycle, Ns, or Q columns.

Recommended file naming:
  GCPL/rate : 1c.mpr  2c.mpr  5c.mpr  10c.mpr  20c.mpr  1c-f.mpr
  CV        : cv.mpr

Requirements:
  pip install galvani numpy pandas matplotlib scipy

Usage:
  python biologic_batch.py
"""

import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# ── try to import galvani (reads .mpr) ────────────────────────────────────────
try:
    from galvani import BioLogic
except ImportError:
    print("[ERROR] galvani not found.  Run:  pip install galvani")
    sys.exit(1)

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
#  FILE LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_mpr(path: str) -> pd.DataFrame:
    """Load an .mpr file → DataFrame with columns t, E, I_mA."""
    mpr = BioLogic.MPRfile(path)
    df  = pd.DataFrame(mpr.data)

    rename = {}
    for col in df.columns:
        cl = col.strip().lower()
        if cl in ("time/s", "time", "t"):
            rename[col] = "t"
        elif cl in ("ewe/v", "ecell/v", "e/v", "voltage", "e"):
            rename[col] = "E"
        elif cl in ("<i>/ma", "i/ma", "current/ma"):
            rename[col] = "I_mA"
        elif cl in ("<i>/a", "i/a", "current/a"):
            rename[col] = "I_A"
    df.rename(columns=rename, inplace=True)

    # unify current → mA
    if "I_A" in df.columns and "I_mA" not in df.columns:
        df["I_mA"] = df["I_A"] * 1000.0

    for req in ("t", "E", "I_mA"):
        if req not in df.columns:
            raise KeyError(
                f"Cannot find '{req}' column in {path}.\n  Available: {list(df.columns)}"
            )

    df = df[["t", "E", "I_mA"]].dropna().reset_index(drop=True)
    return df


def is_cv(filename: str) -> bool:
    return "cv" in os.path.basename(filename).lower()


# ══════════════════════════════════════════════════════════════════════════════
#  HALF-CYCLE SEGMENTATION  (pure t / E / I)
# ══════════════════════════════════════════════════════════════════════════════

def segment_halfcycles(df: pd.DataFrame, smooth_window: int = 11) -> list:
    """
    Split df into half-cycle segments using smoothed-current sign changes.
    Returns a list of DataFrames.
    """
    I    = df["I_mA"].values.copy()
    w    = min(smooth_window, max(3, len(I) // 20))
    I_sm = uniform_filter1d(I, size=w)

    signs = np.sign(I_sm)
    signs[signs == 0] = 1          # treat zero-current as positive

    flip       = np.where(np.diff(signs) != 0)[0] + 1
    boundaries = np.concatenate([[0], flip, [len(df)]])

    segments = []
    for i in range(len(boundaries) - 1):
        seg = df.iloc[boundaries[i]:boundaries[i + 1]].copy()
        if len(seg) > 2:
            segments.append(seg)
    return segments


def get_cycle(segments: list, cycle_idx: int):
    """Return (seg_first_half, seg_second_half) for cycle_idx (1-based)."""
    n_cycles = len(segments) // 2
    if cycle_idx < 1 or cycle_idx > n_cycles:
        raise ValueError(
            f"Cycle {cycle_idx} requested but only {n_cycles} complete cycle(s) found."
        )
    i = (cycle_idx - 1) * 2
    return segments[i], segments[i + 1]


# ══════════════════════════════════════════════════════════════════════════════
#  CAPACITY / dQ/dV
# ══════════════════════════════════════════════════════════════════════════════

def build_specific_capacity(seg: pd.DataFrame, mass_g: float) -> np.ndarray:
    """Cumulative specific capacity array (mAh g⁻¹) for one half-cycle."""
    t  = seg["t"].values
    I  = seg["I_mA"].values
    dt = np.diff(t, prepend=t[0])
    Q  = np.cumsum(np.abs(I) * dt) / 3600.0   # mAh
    return Q / mass_g                           # mAh g⁻¹


def total_specific_capacity(seg: pd.DataFrame, mass_g: float) -> float:
    return float(build_specific_capacity(seg, mass_g)[-1])


def compute_dqdv(seg: pd.DataFrame, mass_g: float, bins: int = 300):
    """dQ/dV vs. V for one half-cycle."""
    Q_sp = build_specific_capacity(seg, mass_g)
    V    = seg["E"].values

    idx  = np.argsort(V)
    V_s  = V[idx];  Q_s = Q_sp[idx]
    V_u  = np.linspace(V_s[0], V_s[-1], bins)
    Q_u  = np.interp(V_u, V_s, Q_s)
    dV   = np.gradient(V_u)
    dQ   = np.gradient(Q_u)
    with np.errstate(divide="ignore", invalid="ignore"):
        dQdV = np.where(np.abs(dV) > 1e-9, dQ / dV, 0.0)
    return V_u, dQdV


# ══════════════════════════════════════════════════════════════════════════════
#  PLOT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

COLORS = {"first":  "#E05A4E", "second": "#4A90D9"}
PLOT_TYPES = ["V-Q", "dQ/dV-V", "V-t", "I-t"]


def _save(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"    Saved: {path}")


def plot_gcpl(df: pd.DataFrame, mass_g: float,
              cycle_idx: int, plot_types: list,
              label: str, out_dir: str) -> dict:
    """Full analysis for one GCPL file."""
    segments = segment_halfcycles(df)
    n_cycles = len(segments) // 2
    print(f"    {label}: {n_cycles} complete cycle(s) detected.")

    if n_cycles == 0:
        print(f"    [SKIP] no complete cycles.")
        return None

    ci = min(cycle_idx, n_cycles)
    if ci != cycle_idx:
        print(f"    [WARN] only {n_cycles} cycle(s) available; using cycle {ci}.")

    seg_a, seg_b = get_cycle(segments, ci)

    # determine which segment is charge vs discharge by majority sign
    sign_a = np.sign(np.median(seg_a["I_mA"].values))
    if sign_a >= 0:
        seg_ch, seg_dc = seg_a, seg_b
    else:
        seg_ch, seg_dc = seg_b, seg_a

    cap_ch = total_specific_capacity(seg_ch, mass_g)
    cap_dc = total_specific_capacity(seg_dc, mass_g)
    ce     = cap_dc / cap_ch * 100 if cap_ch > 0 else float("nan")

    summary = {
        "file":                  label,
        "type":                  "GCPL",
        "cycle_plotted":         ci,
        "n_cycles_total":        n_cycles,
        "charge_cap_mAh_g":      round(cap_ch, 3),
        "discharge_cap_mAh_g":   round(cap_dc, 3),
        "coulombic_eff_%":       round(ce, 2),
    }

    for pt in plot_types:
        pt_safe = pt.replace("/", "_").replace(" ", "")
        base    = os.path.join(out_dir, f"{label}_cy{ci}_{pt_safe}")
        fig, ax = plt.subplots(figsize=(6, 4))

        if pt == "V-Q":
            Qsp_ch = build_specific_capacity(seg_ch, mass_g)
            Qsp_dc = build_specific_capacity(seg_dc, mass_g)
            ax.plot(Qsp_ch, seg_ch["E"].values,
                    color=COLORS["first"],  label="Charge",    lw=1.5)
            ax.plot(Qsp_dc, seg_dc["E"].values,
                    color=COLORS["second"], label="Discharge", lw=1.5)
            ax.set_xlabel("Specific Capacity (mAh g⁻¹)")
            ax.set_ylabel("Voltage (V)")
            ax.set_title(f"{label}  –  V vs. Specific Capacity  (cycle {ci})")
            ax.legend()
            pd.DataFrame({"Specific_Capacity_mAhg": Qsp_ch,
                          "Voltage_V": seg_ch["E"].values}).to_csv(base + "_charge.csv", index=False)
            pd.DataFrame({"Specific_Capacity_mAhg": Qsp_dc,
                          "Voltage_V": seg_dc["E"].values}).to_csv(base + "_discharge.csv", index=False)

        elif pt == "dQ/dV-V":
            Vc, dQc = compute_dqdv(seg_ch, mass_g)
            Vd, dQd = compute_dqdv(seg_dc, mass_g)
            ax.plot(Vc, dQc, color=COLORS["first"],  label="Charge",    lw=1.5)
            ax.plot(Vd, dQd, color=COLORS["second"], label="Discharge", lw=1.5)
            ax.set_xlabel("Voltage (V)")
            ax.set_ylabel("dQ/dV (mAh g⁻¹ V⁻¹)")
            ax.set_title(f"{label}  –  dQ/dV vs. V  (cycle {ci})")
            ax.legend()
            pd.DataFrame({"Voltage_V": Vc, "dQdV": dQc}).to_csv(base + "_charge.csv",    index=False)
            pd.DataFrame({"Voltage_V": Vd, "dQdV": dQd}).to_csv(base + "_discharge.csv", index=False)

        elif pt == "V-t":
            tc = (seg_ch["t"].values - seg_ch["t"].values[0]) / 3600
            td = (seg_dc["t"].values - seg_dc["t"].values[0]) / 3600
            ax.plot(tc, seg_ch["E"].values, color=COLORS["first"],  label="Charge",    lw=1.5)
            ax.plot(td, seg_dc["E"].values, color=COLORS["second"], label="Discharge", lw=1.5)
            ax.set_xlabel("Time (h)")
            ax.set_ylabel("Voltage (V)")
            ax.set_title(f"{label}  –  V vs. t  (cycle {ci})")
            ax.legend()
            pd.DataFrame({"Time_h": tc, "Voltage_V": seg_ch["E"].values}).to_csv(base + "_charge.csv",    index=False)
            pd.DataFrame({"Time_h": td, "Voltage_V": seg_dc["E"].values}).to_csv(base + "_discharge.csv", index=False)

        elif pt == "I-t":
            tc = (seg_ch["t"].values - seg_ch["t"].values[0]) / 3600
            td = (seg_dc["t"].values - seg_dc["t"].values[0]) / 3600
            ax.plot(tc, seg_ch["I_mA"].values, color=COLORS["first"],  label="Charge",    lw=1.5)
            ax.plot(td, seg_dc["I_mA"].values, color=COLORS["second"], label="Discharge", lw=1.5)
            ax.set_xlabel("Time (h)")
            ax.set_ylabel("Current (mA)")
            ax.set_title(f"{label}  –  I vs. t  (cycle {ci})")
            ax.legend()
            pd.DataFrame({"Time_h": tc, "Current_mA": seg_ch["I_mA"].values}).to_csv(base + "_charge.csv",    index=False)
            pd.DataFrame({"Time_h": td, "Current_mA": seg_dc["I_mA"].values}).to_csv(base + "_discharge.csv", index=False)

        _save(fig, base + ".png")

    return summary


def plot_cv(df: pd.DataFrame, label: str, out_dir: str):
    """Plot I vs E for a CV file."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df["E"].values, df["I_mA"].values, lw=1.2, color="#5B4FCF")
    ax.axhline(0, color="gray", lw=0.6, ls="--")
    ax.set_xlabel("Voltage (V)")
    ax.set_ylabel("Current (mA)")
    ax.set_title(f"{label}  –  CV")
    base = os.path.join(out_dir, f"{label}_CV")
    _save(fig, base + ".png")
    pd.DataFrame({"Voltage_V": df["E"].values,
                  "Current_mA": df["I_mA"].values}).to_csv(base + ".csv", index=False)


# ══════════════════════════════════════════════════════════════════════════════
#  USER INPUT HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def ask(prompt: str, default: str = "") -> str:
    val = input(prompt).strip()
    return val if val else default


def choose_plot_types() -> list:
    print("\n  Available plot types for GCPL:")
    for i, pt in enumerate(PLOT_TYPES, 1):
        print(f"    {i}. {pt}")
    raw = ask("  Enter numbers (comma-separated, e.g. 1,2) [default: 1]: ", "1")
    chosen = []
    for tok in raw.split(","):
        tok = tok.strip()
        if tok.isdigit() and 1 <= int(tok) <= len(PLOT_TYPES):
            chosen.append(PLOT_TYPES[int(tok) - 1])
    return chosen if chosen else [PLOT_TYPES[0]]


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 60)
    print("  BioLogic .mpr Batch Analysis")
    print("  Raw columns only: t | E | I")
    print("=" * 60)

    folder = ask("\nData folder path: ")
    if not os.path.isdir(folder):
        print(f"[ERROR] Folder not found: {folder}")
        sys.exit(1)

    mass_mg = float(ask("Active material mass (mg): "))
    mass_g  = mass_mg / 1000.0

    use_specific = ask("\nPlot a specific cycle? (y/n) [default: n]: ", "n").lower()
    if use_specific == "y":
        cycle_idx = int(ask("  Which cycle (1-based)? [default: 2]: ", "2"))
    else:
        cycle_idx = 2

    plot_types = choose_plot_types()

    out_dir = os.path.join(folder, "analysis_output")
    os.makedirs(out_dir, exist_ok=True)
    print(f"\nOutput folder: {out_dir}\n")

    mpr_files = sorted(f for f in os.listdir(folder) if f.lower().endswith(".mpr"))
    if not mpr_files:
        print("[ERROR] No .mpr files found.")
        sys.exit(1)
    print(f"Found {len(mpr_files)} .mpr file(s): {mpr_files}\n")

    summaries = []

    for fname in mpr_files:
        fpath = os.path.join(folder, fname)
        label = os.path.splitext(fname)[0]
        print(f"► {fname}")

        try:
            df = load_mpr(fpath)
        except Exception as exc:
            print(f"  [ERROR] {exc}")
            continue

        if is_cv(fname):
            print("  Detected: CV")
            plot_cv(df, label, out_dir)
            summaries.append({"file": label, "type": "CV"})
        else:
            print("  Detected: GCPL / Rate")
            result = plot_gcpl(df, mass_g, cycle_idx, plot_types, label, out_dir)
            if result:
                summaries.append(result)
        print()

    if summaries:
        df_sum   = pd.DataFrame(summaries)
        sum_path = os.path.join(out_dir, "summary.csv")
        df_sum.to_csv(sum_path, index=False)
        print(f"Summary saved: {sum_path}")
        print("\n" + df_sum.to_string(index=False))

    print("\nAll done.")


if __name__ == "__main__":
    main()

  BioLogic .mpr Batch Analysis
  Raw columns only: t | E | I



Data folder path:  /Users/leili/Downloads/lto-0109
Active material mass (mg):  0.0109

Plot a specific cycle? (y/n) [default: n]:  n



  Available plot types for GCPL:
    1. V-Q
    2. dQ/dV-V
    3. V-t
    4. I-t


  Enter numbers (comma-separated, e.g. 1,2) [default: 1]:  1



Output folder: /Users/leili/Downloads/lto-0109/analysis_output

Found 8 .mpr file(s): ['10c.mpr', '1c-f.mpr', '1c.mpr', '20c.mpr', '2c.mpr', '5c.mpr', 'LL0109-LTO-0_08_OCV_C05.mpr', 'LL0109-LTO-0_09_CV_C05.mpr']

► 10c.mpr
  [ERROR] "Cannot find 'I_mA' column in /Users/leili/Downloads/lto-0109/10c.mpr.\n  Available: ['flags', 'Ns', 't', 'dq/mA.h', '(Q-Qo)/mA.h', 'control/V/mA', 'E', 'I Range', 'Q charge/discharge/mA.h', 'half cycle']"
► 1c-f.mpr
  [ERROR] "Cannot find 'I_mA' column in /Users/leili/Downloads/lto-0109/1c-f.mpr.\n  Available: ['flags', 'Ns', 't', 'dq/mA.h', '(Q-Qo)/mA.h', 'control/V/mA', 'E', 'I Range', 'Q charge/discharge/mA.h', 'half cycle']"
► 1c.mpr
  [ERROR] "Cannot find 'E' column in /Users/leili/Downloads/lto-0109/1c.mpr.\n  Available: ['flags', 't', '<Ewe>/V']"
► 20c.mpr
  [ERROR] "Cannot find 'I_mA' column in /Users/leili/Downloads/lto-0109/20c.mpr.\n  Available: ['flags', 'Ns', 't', 'dq/mA.h', '(Q-Qo)/mA.h', 'control/V/mA', 'E', 'I Range', 'Q charge/discharge/m